# 🧠 Day 22: Gemini SDK, Embeddings, Vector DBs & Document Processing (Hinglish)

Welcome to the Day 22 Practical Notebook! Aaj hum log latest Gemini SDK, text embeddings, semantic search, vector databases, aur LangChain document loaders & splitters ko check karenge.

Is notebook mein:
1. **Gemini Latest SDK Code** (Content Generation)
2. **Gemini Embeddings API Demo**
3. **Semantic Search Simulation** (Sentence Transformers + Cosine Similarity)
4. **LangChain Document Processing** (Loading & Splitting files programmatically)

--- 
## 🚀 1. Latest Gemini SDK Demo

Google ke new standard SDK `google-genai` package ko initialize karke model se text content generate karwana.

In [ ]:
# Install latest GenAI SDK if not present
# !pip install google-genai

from google import genai

# Note: Replace with your actual API key to execute
try:
    client = genai.Client(api_key=" enter your key")
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents="Explain AI in one sentence."
    )
    print("🤖 Response:", response.text)
except Exception as e:
    print("Error or API Key placeholder check:", str(e))

--- 
## 🧠 2. Gemini Embeddings Demo

Text ko multi-dimensional vectors (embeddings) mein convert karne ke liye `client.models.embed_content` function call karte hain.

In [ ]:
try:
    client = genai.Client(api_key="Enter your API Key")
    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents="Python is a programming language."
    )
    print("📏 Vector Dimension size:", len(result.embeddings.values))
    print("🔢 Sample Embedding values (first 5 dimensions):", result.embeddings.values[:5])
except Exception as e:
    print("Gemini Embeddings error (Keys/SDK check):", str(e))

--- 
## 📊 3. Semantic Search using Sentence Transformers

Hum sentences ko encode karke vectors banayenge aur check karenge ki user query ke sabse conceptual close wordings kaun si hain.

In [ ]:
# Dependencies install code if needed
# !pip install sentence-transformers scikit-learn

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Sample documents
documents = [
    "Python is a programming language.",
    "Machine Learning is a branch of AI.",
    "Football is a popular sport.",
    "Cats are friendly animals."
]

print("📚 Documents loaded:")
for idx, doc in enumerate(documents):
    print(f"  [{idx}] {doc}")

# Load sentence transformer model
print("\nLoading embedding model (all-MiniLM-L6-v2)...")
model = SentenceTransformer("all-MiniLM-L6-v2")

# Vectorize documents
document_embeddings = model.encode(documents)

# Mock User query (You can replace this query to test different topics)
query = "Tell me about coding"
print(f"\n🔍 User Query: '{query}'")

query_embedding = model.encode([query])

# Similarity matrix calculation
similarities = cosine_similarity(query_embedding, document_embeddings)
best_match = similarities.argmax()

print("\n🏆 Semantic Search Results:")
print("  Best Match Document:", documents[best_match])
print(f"  Cosine Similarity Score: {similarities[0][best_match]:.4f}")

--- 
## 📄 4. LangChain Document Loading & Processing

RAG pipelines mein text loaders use karke files ko split and chunking kiya jata hai.

### Step A: Programmatic Dummy Data Files Setup

Loaders test karne ke liye hum runtime par placeholder txt aur csv files auto-generate kar rahe hain.

In [ ]:
# Create students.csv
with open("students.csv", "w", encoding="utf-8") as f:
    f.write("Name,Age,City\nAli,22,Jaipur\nSara,25,Delhi\nRaj,23,Mumbai\n")
print("✅ Created students.csv")

# Create notes.txt
with open("notes.txt", "w", encoding="utf-8") as f:
    f.write("Vector database stores embeddings vectors and performs similarity search.\n"
            "SQL database stores rows and columns for exact queries.\n"
            "RAG pipelines connect document loaders to split text chunk vectors.\n")
print("✅ Created notes.txt")

### Step B: Combine Loaders and Text Splitters

In [ ]:
# Install packages if not present
# !pip install langchain langchain-community langchain-text-splitters pypdf

from langchain_community.document_loaders import (
    PyPDFLoader,
    CSVLoader,
    TextLoader
)
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_documents = []
# Load PDF safely if it exists
if os.path.exists("python.pdf"):
    try:
        pdf_loader = PyPDFLoader("python.pdf")
        pdf_documents = pdf_loader.load()
        print(f"Loaded {len(pdf_documents)} pages from python.pdf")
    except Exception as e:
        print("Error loading python.pdf:", e)
else:
    print("python.pdf not found (skipping PDF loading)")

# Load CSV
csv_loader = CSVLoader("students.csv")
csv_documents = csv_loader.load()
print(f"Loaded {len(csv_documents)} records from students.csv")

# Load TXT
txt_loader = TextLoader("notes.txt")
txt_documents = txt_loader.load()
print(f"Loaded text contents from notes.txt")

# Combine all parsed documents
documents = pdf_documents + csv_documents + txt_documents
print(f"\nTotal Documents Loaded: {len(documents)}")

# Split documents into chunks using RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,    # Kept small for visualization
    chunk_overlap=30
)

chunks = splitter.split_documents(documents)
print(f"Total Chunks Created: {len(chunks)}")

# Print out the chunks
for i, chunk in enumerate(chunks, start=1):
    print("=" * 50)
    print(f"Chunk {i}")
    print("=" * 50)
    print(chunk.page_content)
    print("Metadata:", chunk.metadata)
    print()